# PlastPrior: source data inspection

This notebook documents the local input collection used to develop PlastPrior. It reports recorded source versions, file availability, and the dimensions of selected source-native tables. It does not perform chemical identity resolution, evidence integration, or prioritisation.

**Unit of observation:** a parsed row in a declared source table. Counts are not unique chemical counts and must not be summed across overlapping sources. The 12 files contain 14 selected tables; other workbook sheets are outside this inspection.

## 1. Environment and input location

Run this notebook from the project directory with the **PlastPrior (Python 3.12)** kernel. Restart the kernel before running all cells. Set `DATA_ROOT` to an alternative location if required; relative paths are resolved from the project root. Inputs are never modified.

In [ ]:
from pathlib import Path
import platform
from importlib.metadata import version

import pandas as pd
from IPython.display import display

if not (Path.cwd() / "plastprior" / "inputs.py").is_file():
    raise RuntimeError("Open the notebook from the PlastPrior project directory.")

from plastprior.inputs import inspect_inputs, read_source_table

DATA_ROOT = Path("data")
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 100)

runtime = pd.DataFrame({
    "component": ["Python", "Platform", "pandas", "numpy", "openpyxl", "xlrd"],
    "version": [platform.python_version(), platform.system(),
                version("pandas"), version("numpy"), version("openpyxl"), version("xlrd")],
})
display(runtime)

## 2. Inspect the declared collection

All reading and validation are implemented in `plastprior/inputs.py`. Worksheet names, zero-based header offsets, required column labels, and recorded provenance are declared in `plastprior/sources.json`. CSV records are counted in chunks; Excel worksheets are read individually. This step may take several minutes. Missing or unreadable inputs remain visible in the report.

In [ ]:
print("Inspecting the local input collection...")
inspection = inspect_inputs(DATA_ROOT)
display(pd.DataFrame({
    "measure": ["Declared source files", "Selected source tables", "Supporting input groups", "Inspection status"],
    "value": [len(inspection.sources), len(inspection.tables), len(inspection.supporting),
              "Passed" if inspection.passed else "Failed"],
}))

## 3. Source inventory and recorded versions

Versions and snapshot dates below are inherited provenance declarations for the supplied files, not assertions about the latest online releases. **Not recorded** means the available provenance does not establish a value. Filesystem modification dates are not used to fill these gaps. Sizes are decimal megabytes.

In [ ]:
display(inspection.sources[[
    "source", "role", "filename", "version", "snapshot_date", "size_mb", "status"
]])

## 4. Source-native table dimensions

**Parsed rows** are the records returned by the configured reader. **Non-empty rows** exclude rows containing only missing or whitespace-only cells. Blank physical CSV lines and trailing empty worksheet rows may be omitted by the reader. Header row numbers in this table are **one-based**. Required labels are checked before dimensions are accepted.

FCChumon contributes two inventory tables. LitChemPlast contributes a chemical table and a separate literature-to-chemical link table. NORMAN SusDat and PubChemLite are identity-reference tables. These distinct observation units should not be pooled.

In [ ]:
display(inspection.tables[[
    "source_id", "table_id", "table_role", "sheet", "header_row",
    "parsed_rows", "nonempty_rows", "columns", "status"
]])

## 5. Supporting inputs

These are **presence checks only** for named files and directories. Available-file totals are recursive and include any local backup files. A passed presence check does not certify the contents of VEGA reports, GIC records, query evidence, or nested production datasets. Those checks will be introduced with their respective readers.

In [ ]:
display(inspection.supporting)

## 6. Provenance notes and file fingerprints

The notes distinguish recorded versions from dates inferred from filenames. References identify the declared resources and have not been independently revalidated by this notebook. SHA-256 fingerprints identify the bytes inspected in this run; no external reference hash is used to certify them.

In [ ]:
display(inspection.sources[["source", "provenance_note", "reference"]])
display(inspection.sources[["source", "sha256"]])

## 7. Inspection outcome

Issues are displayed before the notebook stops. Missing inputs and invalid table schemas must be resolved before this collection is used downstream. Unrecorded version metadata remains an explicit provenance limitation even when all files are readable.

Workbook-reader warnings are listed separately. Unsupported Excel formatting extensions are not interpreted, and formula values rely on existing workbook caches. The input workbooks are never saved or modified.

In [ ]:
reader_notes = inspection.tables.loc[inspection.tables["reader_warnings"].ne(""), ["table_id", "reader_warnings"]]
if not reader_notes.empty:
    print("Workbook-reader notes")
    display(reader_notes)

for label, frame in [("Source files", inspection.sources),
                     ("Source tables", inspection.tables),
                     ("Supporting inputs", inspection.supporting)]:
    failures = frame.loc[frame["status"].ne("OK")]
    if not failures.empty:
        print(label)
        display(failures)
inspection.require_complete()
print("Input inspection passed within the stated scope.")

## 8. Reading an individual table

The same reader can return a source-native table for later analysis. This example reads at most five ICCA records and reports only its dimensions. It does not standardise identifiers or deduplicate records. For multi-table workbooks, supply an explicit `table_id`.

In [ ]:
example = read_source_table("ICCA", data_root=DATA_ROOT, nrows=5)
print(f"ICCA preview dimensions: {len(example)} rows and {len(example.columns)} columns.")

## Reproducibility boundary

This notebook establishes file readability, basic schema checks, and descriptive table dimensions. It does not establish unique chemical counts, data reuse permissions, scientific validity of individual observations, or complete readiness for scoring. The source notebook is maintained without saved outputs. Executed notebooks and HTML reading copies belong in the ignored `results` directory; they describe the input bytes and runtime shown in that execution.